# Week 14: STT 및 TTS 실무와 음성 멀티모달 파이프라인 (Theme 53)

본 실습 노트북에서는 LLM 애플리케이션의 핵심 인터페이스 중 하나인 **음성(Speech)** 처리 기법을 학습합니다. 
기존의 텍스트 기반 인터페이스에서 벗어나 음성을 텍스트로 전사하는 **STT(Speech-to-Text)**, 텍스트를 고품질 음성으로 합성하는 **TTS(Text-to-Speech)** 기술을 유기적으로 결합하여 최종적으로 실무 프로덕션 레벨의 **Voice-to-Voice 파이프라인**을 완성합니다.

## 🎓 학습 목표
1. **OpenAI Whisper API 실무**: 산업 표준 고품질 음성 인식 모델(Whisper)을 활용하여 한국어 음성 데이터를 정확히 전사하는 방법을 실습합니다.
2. **Google Gemini Multimodal Audio 이해**: 별도의 STT 단계를 거치지 않고 오디오 데이터(`.wav`, `.mp3`)를 직접 Gemini 모델에 주입하여 의미론적인 맥락 및 분위기를 분석하는 LMM 오디오 네이티브 처리를 학습합니다.
3. **OpenAI TTS API 실무**: `tts-1` 모델과 다양한 음성 캐릭터(`nova`, `shimmer`, `alloy` 등)를 조합해 자연스러운 한국어 문장을 합성하고 Jupyter 내에서 재생(IPython.display.Audio)해 봅니다.
4. **Production-Level Voice-to-Voice Pipeline**: 음성 입력 -> STT -> Pandera 규격 검증 및 예외 처리 -> LLM(Gemini) 추론 -> Pandera 답변 검증 -> TTS 합성 및 재생으로 이어지는 견고한 비즈니스 파이프라인을 구축합니다.

---

## 1. 환경 설정 및 API 키 로드

실습에 필요한 라이브러리를 바인딩하고 프로젝트 루트의 `.env` 파일로부터 API 키를 로드합니다.

In [ ]:
import os
import sys
from pathlib import Path
import IPython.display as ipd

# 현재 작업 디렉토리(week14)의 상위(프로젝트 루트)를 Python path에 추가하여 config 모듈 접근성 확보
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from config import GOOGLE_AI_API_KEY, OPENAI_API_KEY, CONTENT_DIR

openai_api_key = OPENAI_API_KEY
gemini_api_key = GOOGLE_AI_API_KEY

print("[상태] OpenAI API 키 로드:", "성공" if openai_api_key else "실패")
print("[상태] Gemini API 키 로드:", "성공" if gemini_api_key else "실패")

## 2. OpenAI Whisper API를 활용한 STT (Speech-to-Text)

오디오 파일을 OpenAI의 `whisper-1` 모델에 전송하여 텍스트로 변환(전사)합니다. 
실습의 완결성을 위해, 먼저 OpenAI TTS를 사용해 실습용 예제 파일(`stt_test_theme53.wav`)을 생성한 후 이를 STT 모델에 전달하는 방식으로 구성합니다.

In [ ]:
from openai import OpenAI

openai_client = OpenAI(api_key=openai_api_key)
test_audio_path = "stt_test_theme53.wav"

try:
    print("[준비] Whisper 실습용 음성 예제 파일을 실시간 생성합니다 (OpenAI TTS 사용)...")
    temp_tts = openai_client.audio.speech.create(
        model="tts-1",
        voice="alloy",
        input="반갑습니다. 오늘 에스케이 패밀리 에이이 캠프 30 과정의 스피치 투 텍스트 실습 수업입니다. 이 음성을 텍스트로 올바르게 변환해 보세요."
    )
    # OpenAI Python SDK v2.43.0+ 표준 파일 저장 방식 사용
    temp_tts.write_to_file(test_audio_path)
    print(f"[성공] 음성 파일 '{test_audio_path}' 생성 완료.")
except Exception as e:
    print("[에러] 예제 파일 생성 실패. 기존 파일이 존재하는지 점검이 필요합니다. 에러:", e)

In [ ]:
try:
    print(f"[실행] '{test_audio_path}' 파일을 Whisper API로 전사합니다...")
    with open(test_audio_path, "rb") as audio_file:
        transcription = openai_client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            language="ko",  # 한국어 힌트를 주어 전사 정확도 향상 및 처리 속도 최적화
            response_format="json"
        )
    print("\n=== Whisper STT 전사 결과 ===")
    print(transcription.text)
except Exception as e:
    print("[에러] Whisper API 호출 실패:", e)

## 3. Google Gemini SDK를 활용한 멀티모달 오디오 직접 분석 (LMM Audio)

최신 멀티모달 LLM(Gemini 2.0/1.5 등)은 텍스트나 이미지뿐만 아니라 오디오 원본 데이터(`.wav`, `.mp3` 등)를 네이티브 데이터로 직접 이해할 수 있습니다.
별도의 STT 파이프라인을 분리하지 않고 오디오 bytes를 직접 LLM에 주입하여 내용 전사 및 분위기, 억양 분석을 통합 처리해 봅니다.

In [ ]:
from google import genai
from google.genai import types

# 공식 google-genai SDK 사용
gemini_client = genai.Client(api_key=gemini_api_key)

try:
    print(f"[실행] '{test_audio_path}' 파일을 Gemini 2.0 Flash API로 전사 및 분위기 분석을 수행합니다...")
    with open(test_audio_path, "rb") as audio_file:
        audio_bytes = audio_file.read()
    
    # 20MB 미만 파일의 경우 types.Part.from_bytes를 사용해 인라인 전달이 경제적이고 빠름
    response = gemini_client.models.generate_content(
        model='gemini-2.0-flash',
        contents=[
            "이 오디오 파일의 내용을 한글 텍스트로 정확히 전사(transcribe)해 주고, 화자의 말에서 느껴지는 어조, 속도, 분위기 등의 시각적/청각적 특징을 분석하여 한글로 정리해 주세요.",
            types.Part.from_bytes(
                data=audio_bytes,
                mime_type='audio/wav'
            )
        ]
    )
    print("\n=== Gemini Multimodal Audio 분석 결과 ===")
    print(response.text)
except Exception as e:
    print("[에러] Gemini API 호출 실패:", e)

## 4. OpenAI TTS API를 활용한 음성 합성 및 재생

텍스트 메시지를 자연스러운 기계 합성음으로 변환하는 TTS 실습을 진행합니다. 
다양한 목소리 캐릭터(`nova`, `shimmer`, `alloy`, `echo`, `fable`, `onyx`)의 특성을 이해하고 합성된 파일을 `IPython.display.Audio` 모듈로 직접 청취해 봅니다.

In [ ]:
tts_output_path = "tts_output_theme53.mp3"
text_to_speak = "반갑습니다! 현업 수준의 음성 에이전트 구축을 도와드릴 시니어 멘토입니다. 오늘 음성 합성 실습 결과가 아주 성공적입니다!"

try:
    print("[실행] OpenAI TTS API 호출 중...")
    response = openai_client.audio.speech.create(
        model="tts-1",
        voice="nova",  # 차분하고 밝은 여성 톤
        input=text_to_speak
    )
    response.write_to_file(tts_output_path)
    print(f"[성공] 음성 합성 파일 '{tts_output_path}' 저장 완료.")
    
    # 주피터 노트북 내 오디오 플레이어 로드
    display(ipd.Audio(tts_output_path))
except Exception as e:
    print("[에러] TTS API 호출 실패:", e)

## 5. Production-Level Voice-to-Voice Pipeline & Pandera 검증

현업 음성 AI 서비스에서는 비정상 입력(침묵, 노이즈, 금지어)과 지나치게 긴 텍스트로 인한 API 비용 폭주를 방어하기 위한 안전성 장치가 필수적입니다.
여기서는 **Pandera**를 이용해 STT 전사 결과(입력) 및 LLM 생성 답변(출력)의 유효성 검증을 거쳐, 안전하게 TTS 합성까지 도달하는 **End-to-End Voice-to-Voice 파이프라인**을 설계하고 실습합니다.

In [ ]:
import pandas as pd
import pandera as pa

# 비즈니스 룰 정의:
# 1. 전사된 텍스트와 생성된 답변은 공백을 제외하고 최소 1글자 이상이어야 합니다.
# 2. 비용 방어를 위해 질문과 답변의 글자 수는 150자로 제한합니다.
# 3. 보안 및 비즈니스 규정상 허용되지 않는 비속어나 금지어 필터링 처리를 수행합니다.
ban_words = ["바보", "멍청이", "해킹"]

pipeline_schema = pa.DataFrameSchema({
    "role": pa.Column(str, pa.Check.isin(["user", "assistant"])),
    "text": pa.Column(str, [
        pa.Check(lambda s: s.str.strip().str.len() > 0, element_wise=True, error="텍스트 데이터가 비어있습니다."),
        pa.Check(lambda s: s.str.len() <= 150, element_wise=True, error="텍스트 길이가 150자 제한을 초과합니다. (비용/성능 초과 방어)"),
        pa.Check(lambda s: ~s.str.contains("|".join(ban_words)), element_wise=True, error="허용되지 않는 금지 단어가 검출되었습니다.")
    ])
})
print("[정보] Pandera 데이터 무결성 검증 스키마 빌드 완료.")

In [ ]:
def run_voice_to_voice_pipeline(input_audio_path: str, output_audio_path: str):
    print(f"\n[시작] Voice-to-Voice 파이프라인 구동 (입력: {input_audio_path})")
    
    # --- STEP 1: STT (Whisper) ---
    try:
        with open(input_audio_path, "rb") as f:
            stt_res = openai_client.audio.transcriptions.create(
                model="whisper-1",
                file=f,
                language="ko"
            )
        user_text = stt_res.text.strip()
        print(f"[STT 완료] 전사 결과: '{user_text}'")
    except Exception as e:
        print("[에러] STT 변환 실패:", e)
        return
        
    # --- STEP 2: Pandera 입력 유효성 검증 ---
    input_df = pd.DataFrame([{"role": "user", "text": user_text}])
    try:
        pipeline_schema.validate(input_df)
        print("[Pandera] 입력 검증 통과 (안정성 기준 부합)")
    except pa.errors.SchemaError as err:
        print(f"[Pandera 검증 실패] 사용자 입력 규칙 위반:\n{err}")
        
        # 예외 가드(Guard clause) 및 Fallback 작동: 에러 안내 메시지를 TTS로 생성
        fallback_msg = "올바르지 않은 음성 질문이 들어왔습니다. 욕설이나 지나치게 긴 질문은 삼가 주세요."
        print(f"[우회] 사용자 경고 음성 합성 개시... ('{fallback_msg}')")
        openai_client.audio.speech.create(
            model="tts-1",
            voice="alloy",
            input=fallback_msg
        ).write_to_file(output_audio_path)
        display(ipd.Audio(output_audio_path))
        return

    # --- STEP 3: LLM 추론 및 컨텍스트 제어 (Gemini 2.0) ---
    try:
        sys_instruction = (
            "너는 친절하고 실용적인 음성 비서야. 답변은 텍스트가 아닌 음성으로 합성되어 송출되므로, "
            "줄글 형태의 구어체로 100자 이내로 간결하고 정확하게 대답해줘."
        )
        response = gemini_client.models.generate_content(
            model='gemini-2.0-flash',
            contents=user_text,
            config=types.GenerateContentConfig(
                system_instruction=sys_instruction,
                temperature=0.3,
                max_output_tokens=150
            )
        )
        assistant_text = response.text.strip()
        print(f"[LLM 완료] 생성된 답변: '{assistant_text}'")
    except Exception as e:
        print("[에러] LLM 생성 실패:", e)
        return

    # --- STEP 4: Pandera 출력 유효성 검증 ---
    output_df = pd.DataFrame([{"role": "assistant", "text": assistant_text}])
    try:
        pipeline_schema.validate(output_df)
        print("[Pandera] 시스템 답변 검증 통과 (답변 안전도 확보)")
    except pa.errors.SchemaError as err:
        print(f"[Pandera 검증 실패] 시스템 답변 규칙 위반:\n{err}")
        # 시스템 출력 오류 시 기본 사과문으로 대체하여 프로덕션 안정성 보호
        assistant_text = "죄송합니다. 답변 생성 중 적절하지 않은 표현이 포함되어 답변을 전달하지 못했습니다. 다시 질문해 주세요."
        
    # --- STEP 5: TTS (OpenAI) ---
    try:
        tts_res = openai_client.audio.speech.create(
            model="tts-1",
            voice="shimmer",
            input=assistant_text
        )
        tts_res.write_to_file(output_audio_path)
        print(f"[TTS 완료] 최종 출력 음성 파일 저장 성공 -> {output_audio_path}")
        display(ipd.Audio(output_audio_path))
    except Exception as e:
        print("[에러] TTS 변환 실패:", e)

### 5.1 정상 작동 케이스 검증

처음에 생성한 올바른 음성 파일(`stt_test_theme53.wav`)을 파이프라인에 입력해 봅니다.

In [ ]:
run_voice_to_voice_pipeline(
    input_audio_path=test_audio_path,
    output_audio_path="pipeline_success_out.mp3"
)

### 5.2 예외 작동 케이스 검증 (금지어 포함)

금지어("멍청이", "바보" 등)가 들어간 질문 음성 파일을 생성한 뒤 파이프라인에 주입해 봅니다. 
Pandera의 입력 유효성 검사에서 검출되고, 파이프라인의 예외 가드(Guard clause)가 작동하여 사용자에게 우회 경고 음성을 리턴하는지 확인합니다.

In [ ]:
invalid_audio_path = "stt_invalid_test.wav"

try:
    print("[준비] 금지 단어가 포함된 테스트 음성 파일을 생성합니다...")
    openai_client.audio.speech.create(
        model="tts-1",
        voice="alloy",
        input="스마트 비서에게 멍청이 라고 부르는 몰상식한 행동을 하면 질문이 통과될까요?"
    ).write_to_file(invalid_audio_path)
    
    # 파이프라인 구동 (Pandera 검증 실패 유도)
    run_voice_to_voice_pipeline(
        input_audio_path=invalid_audio_path,
        output_audio_path="pipeline_invalid_out.mp3"
    )
except Exception as e:
    print("[에러] 예외 케이스 생성 에러:", e)